# Hamburg Bike Counts — Hourly Aggregation

Loads all raw bike-count CSV files from a folder, merges them, audits for
duplicates, then aggregates 5-minute readings into **hourly sums**.

| Step | Output |
|------|--------|
| 1 · Discover & load CSVs | `df_raw` — combined raw table |
| 2 · Duplicate audit | Report + clean `df_clean` |
| 3 · Detect source resolution | Which stations are 5-min vs daily |
| 4 · Aggregate to hourly | `df_hourly` — one row per station·hour |
| 5 · Save output CSV | `hamburg_bike_counts_hourly.csv` |
| 6 · Visualisation | Heatmap + per-station line charts |

**Kaggle paths (default)**
```
INPUT_DIR  = "/kaggle/input/hamburg-bike-timeseries"  # folder with raw CSVs
OUTPUT_DIR = "/kaggle/working"
```
Change `INPUT_DIR` to a local path when running outside Kaggle.

In [2]:
from pathlib import Path
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
# Kaggle: upload your CSV files as a dataset named 'hamburg-bike-timeseries'
# Local : change INPUT_DIR to the folder that holds your CSV files
if Path("csv/").exists():
    # Running on Kaggle
    INPUT_DIR  = Path("csv/")
    OUTPUT_DIR = Path("output/")
else:
    # Running locally — edit these paths
    INPUT_DIR  = Path(".")          # folder containing the raw CSV files
    OUTPUT_DIR = Path(".")          # where the output CSV will be written

OUTPUT_CSV = OUTPUT_DIR / "hamburg_bike_counts_hourly.csv"

# ── Duplicate-detection keys ───────────────────────────────────────────────────
# A record is a duplicate when ALL three of these columns match.
# Using datastream_id + station_name + phenomenon_time is stricter than
# station_name + phenomenon_time alone: it catches cases where two CSVs
# contain the same physical station but with different datastream IDs.
DEDUP_KEYS = ["datastream_id", "station_name", "phenomenon_time"]

# ── Aggregation interval ───────────────────────────────────────────────────────
# For 5-min source data: 12 readings per hour  (60 / 5 = 12)
# For daily source data: already one row per day, handled separately
READINGS_PER_HOUR_5MIN  = 12   # 5-min sensor
READINGS_PER_HOUR_DAILY = 1    # daily sensor (passed through unchanged)

# ── Minimum coverage threshold ─────────────────────────────────────────────────
# Hours with fewer than this fraction of expected readings are flagged.
# 0.5 = at least 6 of 12 readings must be present for a 5-min station.
MIN_COVERAGE = 0.5

print(f"INPUT_DIR  : {INPUT_DIR.resolve()}")
print(f"OUTPUT_CSV : {OUTPUT_CSV.resolve()}")
print(f"Dedup keys : {DEDUP_KEYS}")
print(f"Min coverage threshold : {MIN_COVERAGE:.0%}")

INPUT_DIR  : /home/webapp/work/work/csv
OUTPUT_CSV : /home/webapp/work/work/output/hamburg_bike_counts_hourly.csv
Dedup keys : ['datastream_id', 'station_name', 'phenomenon_time']
Min coverage threshold : 50%


In [4]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

# ── Discover all CSV files in INPUT_DIR ────────────────────────────────────────
csv_files = sorted(INPUT_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s) in {INPUT_DIR}:")
for f in csv_files:
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name:55s}  {size_mb:6.2f} MB")

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {INPUT_DIR}.\n"
        "On Kaggle: add your dataset to the notebook via Add Data → "
        "select your dataset.\n"
        "Locally  : set INPUT_DIR to the folder that holds your CSV files."
    )

Found 3 CSV file(s) in csv:
  hamburg_bike_timeseries_210-240.csv                      567.11 MB
  hamburg_bike_timeseries_270-300.csv                      563.42 MB
  hamburg_bike_timeseries_300-328.csv                       39.00 MB


In [3]:
# ── Load and append all CSVs ──────────────────────────────────────────────────
frames = []
load_report = []

for f in csv_files:
    try:
        df_f = pd.read_csv(f, low_memory=False)
        n_rows = len(df_f)
        n_stations = df_f['station_name'].nunique() if 'station_name' in df_f.columns else '?'
        frames.append(df_f)
        load_report.append({"file": f.name, "rows": n_rows,
                             "stations": n_stations, "status": "OK"})
        print(f"  ✓  {f.name:50s}  {n_rows:>8,} rows  {n_stations} station(s)")
    except Exception as e:
        load_report.append({"file": f.name, "rows": 0,
                             "stations": 0, "status": str(e)})
        print(f"  ✗  {f.name:50s}  FAILED: {e}")

df_raw = pd.concat(frames, ignore_index=True)
print(f"\n✓ Combined raw table : {len(df_raw):>10,} rows")
print(f"  Unique stations    : {df_raw['station_name'].nunique():>10,}")
print(f"  Columns            : {list(df_raw.columns)}")
df_raw.head()

NameError: name 'csv_files' is not defined

In [ ]:
# ── Normalise column types ────────────────────────────────────────────────────
df_raw['phenomenon_time'] = pd.to_datetime(
    df_raw['phenomenon_time'], utc=True, errors='coerce'
)
df_raw['bike_count']    = pd.to_numeric(df_raw['bike_count'],    errors='coerce')
df_raw['datastream_id'] = pd.to_numeric(df_raw['datastream_id'], errors='coerce')
df_raw['layer_name']    = df_raw['layer_name'].fillna('').astype(str)

# Drop rows where timestamp or station name is missing
n_before = len(df_raw)
df_raw = df_raw.dropna(subset=['phenomenon_time', 'station_name'])
n_dropped = n_before - len(df_raw)
if n_dropped:
    print(f"⚠ Dropped {n_dropped:,} rows with missing timestamp or station name")

print(f"Rows after type normalisation : {len(df_raw):,}")
print(f"Timestamp range : {df_raw['phenomenon_time'].min()} → "
      f"{df_raw['phenomenon_time'].max()}")
print(f"\nNull counts per column:")
print(df_raw.isnull().sum().to_string())

## 2 · Duplicate Audit

A duplicate row is one where **`datastream_id` + `station_name` + `phenomenon_time`**
all match another row. This catches:
- The same CSV file uploaded twice in `INPUT_DIR`
- Overlapping date ranges across multiple CSV files from the same station
- The same observation returned under different filenames

The audit runs **before** any aggregation so you see the raw duplicate rate.

In [ ]:
# ── Step 1: identify duplicates ───────────────────────────────────────────────
dup_mask = df_raw.duplicated(subset=DEDUP_KEYS, keep=False)
df_dupes = df_raw[dup_mask].copy()
n_total  = len(df_raw)
n_dupes  = dup_mask.sum()
n_unique_dup_keys = df_dupes.drop_duplicates(subset=DEDUP_KEYS).shape[0]

print("═" * 60)
print(f"  DUPLICATE AUDIT REPORT")
print("═" * 60)
print(f"  Dedup keys             : {DEDUP_KEYS}")
print(f"  Total raw rows         : {n_total:>10,}")
print(f"  Duplicate rows (total) : {n_dupes:>10,}  "
      f"({n_dupes/n_total*100:.2f}% of all rows)")
print(f"  Unique duplicate keys  : {n_unique_dup_keys:>10,}")
print("═" * 60)

if n_dupes == 0:
    print("  ✓  No duplicates found.")
else:
    # ── Per-station breakdown ──────────────────────────────────────────────────
    print(f"\n  Breakdown by station:")
    dup_by_station = (
        df_dupes.groupby(['station_name', 'datastream_id'])
        .size()
        .reset_index(name='duplicate_rows')
        .sort_values('duplicate_rows', ascending=False)
    )
    for _, r in dup_by_station.iterrows():
        pct = r['duplicate_rows'] / n_total * 100
        print(f"    {r['station_name']!r:48s}  "
              f"DS={int(r['datastream_id'])}  "
              f"{int(r['duplicate_rows']):>8,} rows  ({pct:.2f}%)")

    # ── Earliest and latest duplicate timestamps per station ──────────────────
    print(f"\n  Duplicate timestamp ranges per station:")
    dup_ts_range = (
        df_dupes.groupby('station_name')['phenomenon_time']
        .agg(['min','max','count'])
        .rename(columns={'min':'first_dup','max':'last_dup','count':'n_dup_ts'})
        .reset_index()
    )
    for _, r in dup_ts_range.iterrows():
        print(f"    {r['station_name']!r:48s}  "
              f"{str(r['first_dup'])[:16]} → {str(r['last_dup'])[:16]}  "
              f"({int(r['n_dup_ts']):,} rows)")

    print(f"\n  Sample duplicate rows (first 10):")
    print(df_dupes.sort_values(DEDUP_KEYS).head(10)
          [['station_name','datastream_id','phenomenon_time','bike_count']]
          .to_string(index=False))

In [ ]:
# ── Step 2: remove duplicates — keep first occurrence ─────────────────────────
# 'first' preserves the row from whichever CSV file was loaded first (alphabetical).
# If the same observation has different bike_count in two files, the first is kept.
df_clean = df_raw.drop_duplicates(subset=DEDUP_KEYS, keep='first').copy()
df_clean = df_clean.sort_values(['station_name', 'datastream_id', 'phenomenon_time'])
df_clean = df_clean.reset_index(drop=True)

rows_removed = len(df_raw) - len(df_clean)
print(f"Rows before dedup : {len(df_raw):>10,}")
print(f"Rows removed      : {rows_removed:>10,}")
print(f"Rows after dedup  : {len(df_clean):>10,}")
print(f"\n✓ df_clean ready — {len(df_clean):,} unique observations")
df_clean.head()